# Model 2: LightGBM using features and Kfold

This notebook trains a LightGBM model using [featured data](C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_DataProcessing\02_Feature_Eng\Feature_eng.ipynb) and Kfold to avoid 

# CNN with Featured data Model

## Table of Contents
1. [Model Choice](#model-choice)
2. [Feature Selection](#feature-selection)
3. [Implementation](#implementation)
4. [Evaluation](#evaluation)

## 0) Imports

In [3]:
import numpy as np
import pandas as pd
import h5py

from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

import lightgbm as lgb


In [4]:
FEAT_PATH = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_DataProcessing\01_Data\02_processed\norm\nights_train_norm_features_1hz.h5"



with h5py.File(FEAT_PATH, "r") as f:
    X_feat = f["X_feat"][:]         # (n_subj, 18000, n_feat)
    y      = f["y_nights"][:]       # (n_subj, 18000)
    subj   = f["subject_ids"][:]    # (n_subj,)
    feature_names = [x.decode() for x in f["feature_names"][:]]

print("X_feat:", X_feat.shape, "y:", y.shape, "subj:", subj.shape)
print("n_features:", len(feature_names))
print("feature_names:", feature_names)

assert X_feat.shape[:2] == y.shape
assert X_feat.shape[0] == len(subj)
assert X_feat.shape[2] == len(feature_names)


X_feat: (22, 18000, 18) y: (22, 18000) subj: (22,)
n_features: 18
feature_names: ['airflow_rms_1s', 'airflow_ptp_1s', 'abd_rms_1s', 'thor_rms_1s', 'snore_rms_1s', 'spo_proxy_mean_1s', 'airflow_baseline_local', 'airflow_drop_pct_c30', 'airflow_time_below30_c30', 'airflow_time_below10_c30', 'effort_sum_c10', 'thor_abd_corr_c30', 'snore_presence_c30', 'spo_min_c30', 'spo_range_c30', 'spo_slope_c30', 'airflow_drop_x_spo_range', 'airflow_lowdur_x_spo_rng']


In [ ]:
#normalizar por sujeito


2) Flatten + groups (para GroupKFold por sujeito)

In [5]:
n_subj, n_sec, n_feat = X_feat.shape

X_flat = X_feat.reshape(-1, n_feat).astype(np.float32)
y_flat = y.reshape(-1).astype(np.int32)
groups = np.repeat(subj, n_sec)

print("Flat:", X_flat.shape, y_flat.shape, groups.shape)
print("Pos rate:", y_flat.mean())


Flat: (396000, 18) (396000,) (396000,)
Pos rate: 0.06872222222222223


3) Métrica oficial: event_based_f1

In [6]:
def jaccard_overlap(events_a, events_b):
    size_a = len(events_a)
    size_b = len(events_b)

    a_start = np.array([e[0] for e in events_a], dtype=float)
    a_end   = np.array([e[1] for e in events_a], dtype=float)
    b_start = np.array([e[0] for e in events_b], dtype=float)
    b_end   = np.array([e[1] for e in events_b], dtype=float)

    a_start = np.array([a_start for _ in range(size_b)])
    a_end   = np.array([a_end   for _ in range(size_b)])
    b_start = np.transpose(np.array([b_start for _ in range(size_a)]))
    b_end   = np.transpose(np.array([b_end   for _ in range(size_a)]))

    len_a = a_end - a_start
    len_b = b_end - b_start

    inter_start = np.maximum(a_start, b_start)
    inter_end   = np.minimum(a_end, b_end)
    intersection = np.maximum(inter_end - inter_start, 0)

    return intersection / (len_a + len_b - intersection + 1e-12)


def extract_events_from_binary_mask(binary_mask, fs=1):
    binary_mask = np.array([0] + list(binary_mask) + [0], dtype=int)
    diff = np.diff(binary_mask)
    starts = np.where(diff == 1)[0] / fs
    ends   = np.where(diff == -1)[0] / fs
    return [(float(s), float(e)) for s, e in zip(starts, ends)]


def compute_tp_fp_fn_for_each_entry(prediction, reference, min_iou=0.3):
    if len(prediction) == 0:
        return 0, 0, len(reference)
    if len(reference) == 0:
        return 0, len(prediction), 0

    iou = jaccard_overlap(prediction, reference)

    TP1 = np.sum(np.max(iou >= min_iou, axis=0))
    TP2 = np.sum(np.max(iou >= min_iou, axis=1))

    true_positive = int(min(TP1, TP2))
    false_positive = int(len(prediction) - true_positive)
    false_negative = int(len(reference) - true_positive)

    return true_positive, false_positive, false_negative


def event_based_f1(y_true, y_pred, min_iou=0.3):
    total_tp = total_fp = total_fn = 0

    for yt, yp in zip(y_true, y_pred):
        true_events = extract_events_from_binary_mask(yt, fs=1)
        pred_events = extract_events_from_binary_mask(yp, fs=1)

        tp, fp, fn = compute_tp_fp_fn_for_each_entry(pred_events, true_events, min_iou=min_iou)
        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-12)
    recall    = total_tp / (total_tp + total_fn + 1e-12)

    if precision == 0 or recall == 0:
        return 0.0
    return float(2 * precision * recall / (precision + recall))



4) Helpers: threshold tuning com métrica oficial

Aqui a avaliação é por sujeito/noite (como sua função espera: lista de vetores 1Hz).

In [7]:
def tune_threshold_event_f1(p_val_by_subj, y_val_by_subj, thr_grid=None, min_iou=0.3):
    if thr_grid is None:
        thr_grid = np.linspace(0.05, 0.95, 19)

    best_thr = None
    best_f1 = -1.0

    for thr in thr_grid:
        y_pred_bin = [(p >= thr).astype(int) for p in p_val_by_subj]
        f1 = event_based_f1(y_val_by_subj, y_pred_bin, min_iou=min_iou)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(thr)

    return best_thr, float(best_f1)


5) Treino CV com GroupKFold (por sujeito)

In [8]:
gkf = GroupKFold(n_splits=5)

lgb_params = dict(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=200,
    reg_lambda=1.0,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

thr_grid = np.linspace(0.05, 0.95, 19)
MIN_IOU = 0.3

rows = []
models = []
oof_prob = np.zeros_like(y_flat, dtype=np.float32)

for fold, (tr_s, va_s) in enumerate(gkf.split(subj, np.zeros(len(subj)), groups=subj), 1):
    tr_subj = subj[tr_s]
    va_subj = subj[va_s]

    tr_idx = np.isin(groups, tr_subj)
    va_idx = np.isin(groups, va_subj)

    X_tr, y_tr = X_flat[tr_idx], y_flat[tr_idx]
    X_va, y_va = X_flat[va_idx], y_flat[va_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_tr, y_tr)
    models.append(model)

    p_va = model.predict_proba(X_va)[:, 1].astype(np.float32)
    oof_prob[va_idx] = p_va

    # métricas por segundo (sanity)
    auc = roc_auc_score(y_va, p_va) if len(np.unique(y_va)) > 1 else np.nan
    ap  = average_precision_score(y_va, p_va) if len(np.unique(y_va)) > 1 else np.nan

    # organizar por sujeito/noite para métrica oficial
    p_by_subj = []
    y_by_subj = []
    for sid in va_subj:
        ix = np.where(groups == sid)[0]
        p_by_subj.append(oof_prob[ix])
        y_by_subj.append(y_flat[ix])

    best_thr, best_f1 = tune_threshold_event_f1(p_by_subj, y_by_subj, thr_grid=thr_grid, min_iou=MIN_IOU)

    # event-F1 com threshold escolhido
    y_pred_bin = [(p >= best_thr).astype(int) for p in p_by_subj]
    f1_evt = event_based_f1(y_by_subj, y_pred_bin, min_iou=MIN_IOU)

    rows.append({
        "fold": fold,
        "AUC_1Hz": auc,
        "AP_1Hz": ap,
        "best_thr": best_thr,
        "event_F1": f1_evt,
        "n_val_subjects": len(va_subj),
        "pos_rate_val": float(np.mean(y_va)),
    })

results = pd.DataFrame(rows)
results


[LightGBM] [Info] Number of positive: 20300, number of negative: 285700
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020370 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3903
[LightGBM] [Info] Number of data points in the train set: 306000, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 22884, number of negative: 283116
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.017124 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3906
[LightGBM] [Info] Number of data points in the train set: 306000, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 24208, number of negative: 299792
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.027903 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3903
[LightGBM] [Info] Number of data points in the train set: 324000, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


...

[LightGBM] [Info] Number of positive: 21915, number of negative: 302085
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.018091 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3905
[LightGBM] [Info] Number of data points in the train set: 324000, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 19549, number of negative: 304451
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016392 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3905
[LightGBM] [Info] Number of data points in the train set: 324000, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,fold,AUC_1Hz,AP_1Hz,best_thr,event_F1,n_val_subjects,pos_rate_val
0,1,0.675968,0.143455,0.45,0.061856,5,0.076822
1,2,0.672660,0.087003,0.20,0.047138,5,0.048111
2,3,0.674344,0.076999,0.35,0.039643,4,0.041750
3,4,0.668861,0.130391,0.35,0.055633,4,0.073597
4,5,0.630571,0.179171,0.25,0.077910,4,0.106458


6) Resumo + threshold global

In [9]:
display(results)

thr_global = float(results["best_thr"].median())
print("Global threshold (median):", thr_global)

print("\nMEAN metrics:")
print(results[["AUC_1Hz","AP_1Hz","event_F1"]].mean())

print("\nSTD metrics:")
print(results[["AUC_1Hz","AP_1Hz","event_F1"]].std())


,fold,AUC_1Hz,AP_1Hz,best_thr,event_F1,n_val_subjects,pos_rate_val
0,1,0.675968,0.143455,0.45,0.061856,5,0.076822
1,2,0.672660,0.087003,0.20,0.047138,5,0.048111
2,3,0.674344,0.076999,0.35,0.039643,4,0.041750
3,4,0.668861,0.130391,0.35,0.055633,4,0.073597
4,5,0.630571,0.179171,0.25,0.077910,4,0.106458


Global threshold (median): 0.35

MEAN metrics:
AUC_1Hz     0.664481
AP_1Hz      0.123404
event_F1    0.056436
dtype: float64

STD metrics:
AUC_1Hz     0.019139
AP_1Hz      0.041950
event_F1    0.014659
dtype: float64


8) Feature importance

In [10]:
imp = np.mean([m.feature_importances_ for m in models], axis=0)
imp_df = pd.DataFrame({"feature": feature_names, "importance": imp}).sort_values("importance", ascending=False)
imp_df.head(25)



,feature,importance
6,airflow_baseline_local,5087.4
13,spo_min_c30,4965.2
14,spo_range_c30,3678.0
7,airflow_drop_pct_c30,3365.2
5,spo_proxy_mean_1s,2557.0
11,thor_abd_corr_c30,2484.0
16,airflow_drop_x_spo_range,2275.6
17,airflow_lowdur_x_spo_rng,1883.8
10,effort_sum_c10,1840.2
8,airflow_time_below30_c30,1729.0


9) (Opcional) salvar artefatos

In [11]:
import os, joblib

SAVE_DIR = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\baseline_lgbm_eventf1"
os.makedirs(SAVE_DIR, exist_ok=True)

joblib.dump(models, os.path.join(SAVE_DIR, "models.pkl"))
joblib.dump(feature_names, os.path.join(SAVE_DIR, "feature_names.pkl"))
joblib.dump({"lgb_params": lgb_params, "thr_global": thr_global, "min_iou": 0.3}, os.path.join(SAVE_DIR, "config.pkl"))
results.to_csv(os.path.join(SAVE_DIR, "cv_results.csv"), index=False)

print("Saved to:", SAVE_DIR)


Saved to: C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\baseline_lgbm_eventf1


In [12]:
import os
import joblib
import h5py
import numpy as np
import pandas as pd

SAVE_DIR = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\2_BaselineModel\baseline_lgbm"
TEST_H5  = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_DataProcessing\01_Data\02_processed\norm\nights_test_norm_features_1hz.h5" # ou o caminho do features_1hz.h5 do test

models = joblib.load(os.path.join(SAVE_DIR, "models_lgbm_groupkfold.pkl"))            # list of LGBM models
feature_names = joblib.load(os.path.join(SAVE_DIR, "feature_names.pkl"))
config = joblib.load(os.path.join(SAVE_DIR, "config.pkl"))

print("n_models:", len(models))
print("n_features:", len(feature_names))
print("config keys:", config.keys())
print(config)


n_models: 5
n_features: 18
config keys: dict_keys(['lgb_params', 'thr_global', 'pp_min_len', 'pp_merge_gap', 'iou_thr'])
{'lgb_params': {'n_estimators': 600, 'learning_rate': 0.03, 'num_leaves': 63, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_samples': 200, 'reg_lambda': 1.0, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1}, 'thr_global': 0.30000000000000004, 'pp_min_len': 10, 'pp_merge_gap': 5, 'iou_thr': 0.1}


In [13]:
with h5py.File(TEST_H5, "r") as f:
    X_feat_test = f["X_feat"][:]          # (n_test, 18000, n_feat)
    subj_test   = f["subject_ids"][:]
    feat_bytes  = f["feature_names"][:]   # bytes

feat_names_h5 = [x.decode("utf-8") for x in feat_bytes]
assert feat_names_h5 == feature_names, "Feature order mismatch! (H5 vs pkl)"

print("X_feat_test:", X_feat_test.shape, "subj_test:", subj_test.shape)


X_feat_test: (22, 18000, 18) subj_test: (22,)


In [14]:
n_test, n_sec, n_feat = X_feat_test.shape
X_flat = X_feat_test.reshape(-1, n_feat)

preds = []
for m in models:
    preds.append(m.predict_proba(X_flat)[:, 1].astype(np.float32))

p_test_flat = np.mean(np.stack(preds, axis=0), axis=0)    # (n_test*n_sec,)
p_test = p_test_flat.reshape(n_test, n_sec)               # (n_test, n_sec)

print("p_test:", p_test.shape, p_test.min(), p_test.max(), p_test.mean())


c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\v

p_test: (22, 18000) 0.00042383932 0.6713218 0.04064655


In [16]:
# ============================================================
# Generate OOF (CV), train fold-ensemble, post-process, save CSV
# Assumes you already have:
#   - X_feat_train: (n_train, n_sec, n_feat)
#   - y_train:      (n_train, n_sec)  binary 0/1
#   - subj_train:   (n_train,)
#   - X_feat_test:  (n_test,  n_sec, n_feat)
#   - subj_test:    (n_test,)
#   - event_based_f1(y_true_list, y_pred_list)  (your function)
# ============================================================

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

# -----------------------------
# 1) Helpers: temporal post-processing
# -----------------------------
def smooth_moving_average(p, win=7):
    """Centered moving average. win must be odd."""
    win = int(win)
    if win <= 1:
        return p
    if win % 2 == 0:
        win += 1
    kernel = np.ones(win, dtype=np.float32) / win
    return np.convolve(p, kernel, mode="same")

def hysteresis_binarize(p, thr_on=0.30, thr_off=0.20):
    """
    Hysteresis: start event when p>=thr_on, end when p<thr_off.
    Returns binary mask (0/1).
    """
    y = np.zeros_like(p, dtype=np.int8)
    on = False
    for i, pi in enumerate(p):
        if not on and pi >= thr_on:
            on = True
        elif on and pi < thr_off:
            on = False
        y[i] = 1 if on else 0
    return y

def remove_short_events(mask, min_len=10):
    """Remove positive runs shorter than min_len seconds."""
    m = mask.astype(np.int8).copy()
    padded = np.r_[0, m, 0]
    d = np.diff(padded)
    starts = np.where(d == 1)[0]
    ends   = np.where(d == -1)[0]
    for s, e in zip(starts, ends):
        if (e - s) < min_len:
            m[s:e] = 0
    return m

def fill_short_gaps(mask, max_gap=5):
    """Fill 0-gaps shorter/equal than max_gap inside positive segments."""
    m = mask.astype(np.int8).copy()
    padded = np.r_[0, m, 0]
    d = np.diff(padded)
    # gaps are runs of zeros between ones: look at runs of 0 in the original mask
    # easier: invert and remove short events in inverted mask, but only for interior gaps
    inv = 1 - m
    inv2 = remove_short_events(inv, min_len=max_gap+1)  # keep only long zero-runs
    # inv2==1 means long gaps, so short gaps became 0 -> fill them
    filled = 1 - inv2
    # ensure edges (leading/trailing gaps) stay gaps
    first_one = np.argmax(m == 1) if np.any(m == 1) else None
    last_one  = (len(m) - 1 - np.argmax(m[::-1] == 1)) if np.any(m == 1) else None
    if first_one is None:
        return m
    filled[:first_one] = 0
    filled[last_one+1:] = 0
    return filled.astype(np.int8)

def post_process_probs(p, smooth_win=7, thr_on=0.30, thr_off=0.20, min_len=10, max_gap=5):
    """
    Simple temporal post-processing pipeline:
      1) smooth probs
      2) hysteresis threshold
      3) fill short gaps
      4) remove short events
    """
    p2 = smooth_moving_average(p, win=smooth_win)
    m  = hysteresis_binarize(p2, thr_on=thr_on, thr_off=thr_off)
    if max_gap and max_gap > 0:
        m = fill_short_gaps(m, max_gap=max_gap)
    if min_len and min_len > 1:
        m = remove_short_events(m, min_len=min_len)
    return m, p2

# -----------------------------
# 2) Flatten helpers
# -----------------------------
def flatten_nights(X_nights):
    n_subj, n_sec, n_feat = X_nights.shape
    X_flat = X_nights.reshape(-1, n_feat)
    return X_flat

def flatten_labels(y_nights):
    return y_nights.reshape(-1).astype(np.int8)

def repeat_subjects(subj_ids, n_sec):
    return np.repeat(subj_ids, n_sec)

# -----------------------------
# 3) CV OOF + fold models
# -----------------------------
def train_lgbm_oof_and_models(
    X_feat_train, y_train, subj_train,
    n_splits=5, seed=42,
    lgb_params=None,
    use_scale_pos_weight=True,
):
    n_subj, n_sec, n_feat = X_feat_train.shape
    X_flat = flatten_nights(X_feat_train)
    y_flat = flatten_labels(y_train)
    groups = repeat_subjects(subj_train, n_sec)

    gkf = GroupKFold(n_splits=n_splits)

    oof_pred = np.zeros_like(y_flat, dtype=np.float32)
    models = []
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_flat, y_flat, groups), start=1):
        X_tr, y_tr = X_flat[tr_idx], y_flat[tr_idx]
        X_va, y_va = X_flat[va_idx], y_flat[va_idx]

        params = dict(
            objective="binary",
            learning_rate=0.03,
            n_estimators=5000,
            num_leaves=64,
            min_data_in_leaf=200,
            feature_fraction=0.8,
            bagging_fraction=0.8,
            bagging_freq=1,
            reg_lambda=1.0,
            random_state=seed + fold,
            n_jobs=-1,
        )
        if lgb_params:
            params.update(lgb_params)

        # handle imbalance per fold (recommended)
        if use_scale_pos_weight:
            pos = y_tr.sum()
            neg = len(y_tr) - pos
            spw = float(neg / max(pos, 1))
            params.pop("class_weight", None)
            params["scale_pos_weight"] = spw

        model = lgb.LGBMClassifier(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            eval_metric="binary_logloss",
            callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)],
        )

        p_va = model.predict_proba(X_va)[:, 1].astype(np.float32)
        oof_pred[va_idx] = p_va

        # metrics (1Hz, per-second)
        auc = roc_auc_score(y_va, p_va) if len(np.unique(y_va)) > 1 else np.nan
        ap  = average_precision_score(y_va, p_va) if len(np.unique(y_va)) > 1 else np.nan

        fold_rows.append({"fold": fold, "AUC_1Hz": auc, "AP_1Hz": ap})

        models.append(model)
        print(f"[fold {fold}] AUC={auc:.4f} AP={ap:.4f} best_iter={model.best_iteration_}")

    return oof_pred, models, pd.DataFrame(fold_rows)

# -----------------------------
# 4) Convert flat preds back to nights
# -----------------------------
def unflatten_to_nights(p_flat, n_subj, n_sec):
    return p_flat.reshape(n_subj, n_sec)

# -----------------------------
# 5) Tune post-processing on OOF (simple grid)
# -----------------------------
def tune_postprocess_on_oof(oof_pred_nights, y_train, grid):
    best = {"event_F1": -1}
    results = []
    for cfg in grid:
        y_hat_list = []
        y_true_list = []
        for p, yt in zip(oof_pred_nights, y_train):
            m, _ = post_process_probs(
                p,
                smooth_win=cfg["smooth_win"],
                thr_on=cfg["thr_on"],
                thr_off=cfg["thr_off"],
                min_len=cfg["min_len"],
                max_gap=cfg["max_gap"],
            )
            y_hat_list.append(m)
            y_true_list.append(yt.astype(np.int8))

        f1 = event_based_f1(y_true_list, y_hat_list)
        row = dict(cfg)
        row["event_F1"] = f1
        results.append(row)
        if f1 > best["event_F1"]:
            best = row
    return best, pd.DataFrame(results).sort_values("event_F1", ascending=False)

# -----------------------------
# 6) Predict test using fold ensemble + post-process
# -----------------------------
def predict_test_ensemble(X_feat_test, models):
    X_test_flat = flatten_nights(X_feat_test)
    preds = []
    for m in models:
        preds.append(m.predict_proba(X_test_flat)[:, 1].astype(np.float32))
    p_mean = np.mean(np.stack(preds, axis=0), axis=0)
    return p_mean

# -----------------------------
# 7) Save submission CSV (formato: 1 linha por janela de 90s)
# -----------------------------
def save_submission_csv(subj_test, y_pred_nights_bin, out_csv="submission.csv"):
    """
    Formato: cada linha = 1 janela de 90 segundos
    Colunas: ID (window_id), y_0, y_1, ..., y_89
    Se há n_test sujeitos com n_sec segundos cada, teremos (n_test * n_sec / 90) linhas
    """
    n_test, n_sec = y_pred_nights_bin.shape
    n_windows_per_subj = n_sec // 90
    
    rows = []
    window_id = 0
    
    for subj_idx in range(n_test):
        for win_idx in range(n_windows_per_subj):
            # Extrair os 90 segundos desta janela
            start_sec = win_idx * 90
            end_sec = start_sec + 90
            window_labels = y_pred_nights_bin[subj_idx, start_sec:end_sec]
            
            # Criar linha: ID + y_0 a y_89
            row_data = {"ID": window_id}
            for j in range(90):
                row_data[f"y_{j}"] = int(window_labels[j])
            
            rows.append(row_data)
            window_id += 1
    
    sub = pd.DataFrame(rows)
    sub.to_csv(out_csv, index=False)
    print("Saved submission:", out_csv)
    print(f"  Shape: {sub.shape} (esperado: {n_test * n_windows_per_subj} linhas × 91 colunas)")
    print(f"  Total de janelas: {window_id} ({n_test} sujeitos × {n_windows_per_subj} janelas)")
    return sub



In [17]:
# Preparar variáveis para treino
# As variáveis já foram carregadas anteriormente com nomes diferentes
X_feat_train = X_feat
y_train = y
subj_train = subj

print("Training data ready:")
print(f"  X_feat_train: {X_feat_train.shape}")
print(f"  y_train: {y_train.shape}")
print(f"  subj_train: {subj_train.shape}")


Training data ready:
  X_feat_train: (22, 18000, 18)
  y_train: (22, 18000)
  subj_train: (22,)


In [18]:
# ============================================================
# RUN
# ============================================================

# 1) Train CV, get OOF + fold models
oof_flat, models, df_cv = train_lgbm_oof_and_models(
    X_feat_train=X_feat_train,
    y_train=y_train,
    subj_train=subj_train,
    n_splits=5,
    seed=42,
    lgb_params={
        # you can add/override here
        # "class_weight": "balanced",  # <-- don't combine with scale_pos_weight
    },
    use_scale_pos_weight=True,
)

print(df_cv)

[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Number of positive: 20300, number of

c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[fold 1] AUC=0.7092 AP=0.1515 best_iter=2
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [

c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[fold 2] AUC=0.6007 AP=0.0747 best_iter=1
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [

c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[fold 3] AUC=0.5825 AP=0.0559 best_iter=1
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [

c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[fold 4] AUC=0.6537 AP=0.1273 best_iter=2
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [

c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [19]:
# 2) OOF back to nights
n_train, n_sec, _ = X_feat_train.shape
oof_nights = unflatten_to_nights(oof_flat, n_train, n_sec)

In [20]:
# 3) Diagnóstico: verificar probabilidades OOF antes do tuning
print("Diagnóstico das probabilidades OOF:")
print(f"  Min: {oof_nights.min():.4f}")
print(f"  Max: {oof_nights.max():.4f}")
print(f"  Mean: {oof_nights.mean():.4f}")
print(f"  Median: {np.median(oof_nights):.4f}")
print(f"  Percentis: 50%={np.percentile(oof_nights, 50):.4f}, 75%={np.percentile(oof_nights, 75):.4f}, 90%={np.percentile(oof_nights, 90):.4f}, 95%={np.percentile(oof_nights, 95):.4f}")

# Testar primeiro com threshold simples para baseline
print("\nTestando thresholds simples (sem post-processing):")
for thr in [0.1, 0.2, 0.3, 0.4, 0.5]:
    y_hat_simple = [(p >= thr).astype(np.int8) for p in oof_nights]
    f1_simple = event_based_f1(list(y_train), y_hat_simple)
    n_events = sum([len(extract_events_from_binary_mask(y)) for y in y_hat_simple])
    print(f"  thr={thr:.2f}: F1={f1_simple:.4f}, n_events={n_events}")

# 3) Tune post-processing com grid mais amplo e menos restritivo
grid = [
    # Menos suavização, thresholds mais baixos, menos restrições
    {"smooth_win": 1, "thr_on": 0.15, "thr_off": 0.10, "min_len": 5, "max_gap": 3},
    {"smooth_win": 1, "thr_on": 0.20, "thr_off": 0.15, "min_len": 5, "max_gap": 3},
    {"smooth_win": 1, "thr_on": 0.25, "thr_off": 0.18, "min_len": 8, "max_gap": 5},
    {"smooth_win": 3, "thr_on": 0.20, "thr_off": 0.15, "min_len": 8, "max_gap": 5},
    {"smooth_win": 3, "thr_on": 0.25, "thr_off": 0.18, "min_len": 10, "max_gap": 5},
    {"smooth_win": 5, "thr_on": 0.25, "thr_off": 0.18, "min_len": 10, "max_gap": 7},
    {"smooth_win": 5, "thr_on": 0.30, "thr_off": 0.22, "min_len": 10, "max_gap": 7},
    {"smooth_win": 7, "thr_on": 0.30, "thr_off": 0.22, "min_len": 10, "max_gap": 7},
    # Variações sem fill_gap (max_gap=0)
    {"smooth_win": 3, "thr_on": 0.20, "thr_off": 0.15, "min_len": 8, "max_gap": 0},
    {"smooth_win": 5, "thr_on": 0.25, "thr_off": 0.18, "min_len": 10, "max_gap": 0},
    # Variações sem min_len muito restritivo
    {"smooth_win": 3, "thr_on": 0.20, "thr_off": 0.15, "min_len": 3, "max_gap": 5},
    {"smooth_win": 5, "thr_on": 0.25, "thr_off": 0.18, "min_len": 5, "max_gap": 5},
]

print("\nTunando pós-processamento no OOF...")
best_cfg, df_pp = tune_postprocess_on_oof(oof_nights, y_train, grid)
print("\nBest post-process:", best_cfg)
print("\nTop 10 configurações:")
print(df_pp.head(10))

# Verificar quantos eventos são gerados pela melhor config
if best_cfg["event_F1"] > 0:
    print("\nVerificando eventos gerados pela melhor config:")
    y_hat_best = []
    for p in oof_nights:
        m, _ = post_process_probs(
            p,
            smooth_win=best_cfg["smooth_win"],
            thr_on=best_cfg["thr_on"],
            thr_off=best_cfg["thr_off"],
            min_len=best_cfg["min_len"],
            max_gap=best_cfg["max_gap"],
        )
        y_hat_best.append(m)
    
    n_pred_events = sum([len(extract_events_from_binary_mask(y)) for y in y_hat_best])
    n_true_events = sum([len(extract_events_from_binary_mask(y)) for y in y_train])
    print(f"  Eventos preditos: {n_pred_events}")
    print(f"  Eventos verdadeiros: {n_true_events}")
else:
    print("\n⚠️ ATENÇÃO: F1 ainda zerado! Possíveis problemas:")
    print("  - Probabilidades muito baixas (verificar calibração do modelo)")
    print("  - Threshold muito alto para os dados")
    print("  - Post-processing muito agressivo")


Diagnóstico das probabilidades OOF:
  Min: 0.0620
  Max: 0.1862
  Mean: 0.0914
  Median: 0.0888
  Percentis: 50%=0.0888, 75%=0.0966, 90%=0.1108, 95%=0.1278

Testando thresholds simples (sem post-processing):
  thr=0.10: F1=0.0790, n_events=8906
  thr=0.20: F1=0.0000, n_events=0
  thr=0.30: F1=0.0000, n_events=0
  thr=0.40: F1=0.0000, n_events=0
  thr=0.50: F1=0.0000, n_events=0

Tunando pós-processamento no OOF...

Best post-process: {'smooth_win': 1, 'thr_on': 0.15, 'thr_off': 0.1, 'min_len': 5, 'max_gap': 3, 'event_F1': 0.08514851485148507}

Top 10 configurações:
   smooth_win  thr_on  thr_off  min_len  max_gap  event_F1
0           1    0.15     0.10        5        3  0.085149
1           1    0.20     0.15        5        3  0.000000
2           1    0.25     0.18        8        5  0.000000
3           3    0.20     0.15        8        5  0.000000
4           3    0.25     0.18       10        5  0.000000
5           5    0.25     0.18       10        7  0.000000
6           5  

In [21]:
# 3.5) Confirmar e ajustar parâmetros finais (se necessário)
print("="*60)
print("PARÂMETROS SELECIONADOS PARA INFERÊNCIA NO TEST:")
print("="*60)
for key, val in best_cfg.items():
    print(f"  {key}: {val}")

# Se quiser sobrescrever manualmente algum parâmetro, descomente e ajuste:
# best_cfg["smooth_win"] = 3
# best_cfg["thr_on"] = 0.20
# best_cfg["thr_off"] = 0.15
# best_cfg["min_len"] = 8
# best_cfg["max_gap"] = 5

print("\nParâmetros finais que serão aplicados ao test set:")
print(best_cfg)


PARÂMETROS SELECIONADOS PARA INFERÊNCIA NO TEST:
  smooth_win: 1
  thr_on: 0.15
  thr_off: 0.1
  min_len: 5
  max_gap: 3
  event_F1: 0.08514851485148507

Parâmetros finais que serão aplicados ao test set:
{'smooth_win': 1, 'thr_on': 0.15, 'thr_off': 0.1, 'min_len': 5, 'max_gap': 3, 'event_F1': 0.08514851485148507}


In [22]:
# 4) Predict test with ensemble
p_test_flat = predict_test_ensemble(X_feat_test, models)
n_test, n_sec_test, _ = X_feat_test.shape
p_test_nights = p_test_flat.reshape(n_test, n_sec_test)

[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_data_in_leaf is set=200, min_

c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\tfamd_gpu\lib\site-packages\sklearn\utils\v

In [57]:
smooth_win = 7
thr_on = 0.10
thr_off = 0.09
min_len = 10
max_gap = 5


In [26]:
# 5) Apply best post-process to test probabilities
y_test_pred_bin = np.zeros((n_test, n_sec_test), dtype=np.int8)
for i in range(n_test):
    m, _ = post_process_probs(
        p_test_nights[i],
        smooth_win=best_cfg["smooth_win"],
        thr_on=best_cfg["thr_on"],
        thr_off=best_cfg["thr_off"],
        min_len=best_cfg["min_len"],
        max_gap=best_cfg["max_gap"],
    )
    y_test_pred_bin[i] = m



In [27]:
# 4.5) Verificar predições ANTES do pós-processamento
print("="*60)
print("ESTATÍSTICAS DAS PROBABILIDADES (ANTES DO PÓS-PROCESSAMENTO):")
print("="*60)

# Estatísticas das probabilidades
print(f"\nProbabilidades:")
print(f"  Min:    {p_test_nights.min():.4f}")
print(f"  Max:    {p_test_nights.max():.4f}")
print(f"  Mean:   {p_test_nights.mean():.4f}")
print(f"  Median: {np.median(p_test_nights):.4f}")
print(f"  Std:    {p_test_nights.std():.4f}")

# Testar com thresholds simples
print(f"\nPredições com threshold simples (SEM pós-processamento):")
for thr in [0.1, 0.2, 0.3, 0.4, 0.5]:
    y_simple = (p_test_nights >= thr).astype(int)
    n_pos = y_simple.sum()
    pct = (n_pos / y_simple.size) * 100
    print(f"  thr={thr:.2f}: {n_pos:6d} positivos ({pct:5.2f}%)")

print(f"\nParâmetros de pós-processamento que serão aplicados:")
print(f"  smooth_win: {best_cfg['smooth_win']}")
print(f"  thr_on:     {best_cfg['thr_on']}")
print(f"  thr_off:    {best_cfg['thr_off']}")
print(f"  min_len:    {best_cfg['min_len']}")
print(f"  max_gap:    {best_cfg['max_gap']}")


ESTATÍSTICAS DAS PROBABILIDADES (ANTES DO PÓS-PROCESSAMENTO):

Probabilidades:
  Min:    0.0690
  Max:    0.1247
  Mean:   0.0945
  Median: 0.0941
  Std:    0.0111

Predições com threshold simples (SEM pós-processamento):
  thr=0.10: 126610 positivos (31.97%)
  thr=0.20:      0 positivos ( 0.00%)
  thr=0.30:      0 positivos ( 0.00%)
  thr=0.40:      0 positivos ( 0.00%)
  thr=0.50:      0 positivos ( 0.00%)

Parâmetros de pós-processamento que serão aplicados:
  smooth_win: 1
  thr_on:     0.15
  thr_off:    0.1
  min_len:    5
  max_gap:    3


In [28]:
# 6) Save submission CSV
sub = save_submission_csv(subj_test, y_test_pred_bin, out_csv="submission_lgbm_features.csv")
sub.head()

Saved submission: submission_lgbm_features.csv
  Shape: (4400, 91) (esperado: 4400 linhas × 91 colunas)
  Total de janelas: 4400 (22 sujeitos × 200 janelas)


,ID,y_0,y_1,y_2,y_3,y_4,y_5,y_6,y_7,y_8,...,y_80,y_81,y_82,y_83,y_84,y_85,y_86,y_87,y_88,y_89
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [29]:
# Verificar estatísticas das predições
print("="*60)
print("ESTATÍSTICAS DAS PREDIÇÕES:")
print("="*60)

# Total de predições
total_preds = y_test_pred_bin.size
n_positives = y_test_pred_bin.sum()
n_negatives = total_preds - n_positives
pct_positives = (n_positives / total_preds) * 100

print(f"\nTotal de predições: {total_preds:,}")
print(f"  Positivas (1): {n_positives:,} ({pct_positives:.2f}%)")
print(f"  Negativas (0): {n_negatives:,} ({100-pct_positives:.2f}%)")

# Por sujeito
print(f"\nPor sujeito:")
for i in range(len(subj_test)):
    n_pos_subj = y_test_pred_bin[i].sum()
    pct_pos_subj = (n_pos_subj / len(y_test_pred_bin[i])) * 100
    print(f"  Sujeito {subj_test[i]}: {n_pos_subj:4d} positivos ({pct_pos_subj:5.2f}%)")

# Estatísticas esperadas (baseado no train)
print(f"\n{'='*60}")
print(f"COMPARAÇÃO COM TREINO:")
print(f"{'='*60}")
print(f"Taxa de apneia no treino: ~6.9%")
print(f"Taxa de apneia no test:   {pct_positives:.2f}%")

if pct_positives < 1.0:
    print("\n⚠️ ATENÇÃO: Taxa muito baixa! Considere ajustar threshold.")
elif pct_positives > 20.0:
    print("\n⚠️ ATENÇÃO: Taxa muito alta! Considere ajustar threshold.")
else:
    print("\n✓ Taxa parece razoável.")


ESTATÍSTICAS DAS PREDIÇÕES:

Total de predições: 396,000
  Positivas (1): 0 (0.00%)
  Negativas (0): 396,000 (100.00%)

Por sujeito:
  Sujeito 0:    0 positivos ( 0.00%)
  Sujeito 1:    0 positivos ( 0.00%)
  Sujeito 2:    0 positivos ( 0.00%)
  Sujeito 3:    0 positivos ( 0.00%)
  Sujeito 4:    0 positivos ( 0.00%)
  Sujeito 5:    0 positivos ( 0.00%)
  Sujeito 6:    0 positivos ( 0.00%)
  Sujeito 7:    0 positivos ( 0.00%)
  Sujeito 8:    0 positivos ( 0.00%)
  Sujeito 9:    0 positivos ( 0.00%)
  Sujeito 10:    0 positivos ( 0.00%)
  Sujeito 11:    0 positivos ( 0.00%)
  Sujeito 12:    0 positivos ( 0.00%)
  Sujeito 13:    0 positivos ( 0.00%)
  Sujeito 14:    0 positivos ( 0.00%)
  Sujeito 15:    0 positivos ( 0.00%)
  Sujeito 16:    0 positivos ( 0.00%)
  Sujeito 17:    0 positivos ( 0.00%)
  Sujeito 18:    0 positivos ( 0.00%)
  Sujeito 19:    0 positivos ( 0.00%)
  Sujeito 20:    0 positivos ( 0.00%)
  Sujeito 21:    0 positivos ( 0.00%)

COMPARAÇÃO COM TREINO:
Taxa de apneia no

In [35]:
# ============================================================
# ALTERNATIVA: USAR THRESHOLD SIMPLES (SEM PÓS-PROCESSAMENTO)
# ============================================================
# Como o pós-processamento zerou tudo, vamos usar threshold simples

# Escolher threshold baseado no OOF (0.10 deu F1=0.07)
simple_threshold = 0.11

print(f"Usando threshold simples: {simple_threshold}")
y_test_pred_simple = (p_test_nights >= simple_threshold).astype(int)

# Estatísticas
total = y_test_pred_simple.size
n_pos = y_test_pred_simple.sum()
pct_pos = 100 * n_pos / total

print(f"\nPredições com threshold simples {simple_threshold}:")
print(f"  Total: {total:,}")
print(f"  Positivos: {n_pos:,} ({pct_pos:.2f}%)")
print(f"  Target esperado: ~6.9%")

# Verificar por sujeito
print(f"\nPor sujeito:")
for i in range(y_test_pred_simple.shape[0]):
    n_pos_subj = y_test_pred_simple[i].sum()
    pct = 100 * n_pos_subj / y_test_pred_simple.shape[1]
    print(f"  Sujeito {i}: {n_pos_subj:5d} positivos ({pct:5.2f}%)")

Usando threshold simples: 0.11

Predições com threshold simples 0.11:
  Total: 396,000
  Positivos: 39,669 (10.02%)
  Target esperado: ~6.9%

Por sujeito:
  Sujeito 0:  1050 positivos ( 5.83%)
  Sujeito 1:  2111 positivos (11.73%)
  Sujeito 2:   808 positivos ( 4.49%)
  Sujeito 3:   170 positivos ( 0.94%)
  Sujeito 4:  1742 positivos ( 9.68%)
  Sujeito 5:  6901 positivos (38.34%)
  Sujeito 6:  1456 positivos ( 8.09%)
  Sujeito 7:   125 positivos ( 0.69%)
  Sujeito 8:  9083 positivos (50.46%)
  Sujeito 9:   704 positivos ( 3.91%)
  Sujeito 10:   258 positivos ( 1.43%)
  Sujeito 11:  1181 positivos ( 6.56%)
  Sujeito 12:  2655 positivos (14.75%)
  Sujeito 13:   361 positivos ( 2.01%)
  Sujeito 14:   794 positivos ( 4.41%)
  Sujeito 15:   375 positivos ( 2.08%)
  Sujeito 16:  1046 positivos ( 5.81%)
  Sujeito 17:   280 positivos ( 1.56%)
  Sujeito 18:   268 positivos ( 1.49%)
  Sujeito 19:  2229 positivos (12.38%)
  Sujeito 20:  6001 positivos (33.34%)
  Sujeito 21:    71 positivos ( 0.39

In [39]:
#fill gap 3
# Fill gaps de até 3 s nas predições com threshold simples
y_test_pred_simple_fg3 = np.zeros_like(y_test_pred_simple, dtype=np.int8)
for i in range(y_test_pred_simple.shape[0]):
    y_test_pred_simple_fg3[i] = fill_short_gaps(y_test_pred_simple[i], max_gap=3)

# Salvar submission com fill-gap 3
sub_simple = save_submission_csv(
    subj_test,
    y_test_pred_simple_fg3,
    out_csv="submission_simple_threshold_fillgap3.csv"
)


Saved submission: submission_simple_threshold_fillgap3.csv
  Shape: (4400, 91) (esperado: 4400 linhas × 91 colunas)
  Total de janelas: 4400 (22 sujeitos × 200 janelas)


In [42]:
# Substituir ID pela coluna do y_benchmark.csv
benchmark_path = r"C:\Users\giuli\Documents\Open_Campus\Sleep_Apnea\00_DataProcessing\00_additional_files_dreem\y_benchmark.csv"
ids_df = pd.read_csv(benchmark_path)

# Garante que existe coluna ID
if "ID" not in ids_df.columns:
    raise ValueError(f"Coluna 'ID' não encontrada em {benchmark_path}. Colunas: {ids_df.columns.tolist()}")

# Verifica tamanho compatível
if len(ids_df) != len(sub_simple):
    raise ValueError(f"Nº de linhas mismatch: benchmark={len(ids_df)} vs submission={len(sub_simple)}")

# Aplica IDs e salva
sub_with_ids = sub_simple.copy()
sub_with_ids["ID"] = ids_df["ID"].values
out_csv = "submission_simple_threshold_fillgap3_with_ids.csv"
sub_with_ids.to_csv(out_csv, index=False)
print(f"Saved with benchmark IDs: {out_csv} (shape={sub_with_ids.shape})")

Saved with benchmark IDs: submission_simple_threshold_fillgap3_with_ids.csv (shape=(4400, 91))


In [40]:
print("="*60)
print("ESTATÍSTICAS DAS PREDIÇÕES (FILL GAP 3s):")
print("="*60)

total_preds = y_test_pred_simple_fg3.size
n_positives = y_test_pred_simple_fg3.sum()
n_negatives = total_preds - n_positives
pct_positives = (n_positives / total_preds) * 100

print(f"\nTotal de predições: {total_preds:,}")
print(f"  Positivas (1): {n_positives:,} ({pct_positives:.2f}%)")
print(f"  Negativas (0): {n_negatives:,} ({100-pct_positives:.2f}%)")

print(f"\nPor sujeito:")
for i in range(len(subj_test)):
    n_pos_subj = y_test_pred_simple_fg3[i].sum()
    pct_pos_subj = (n_pos_subj / y_test_pred_simple_fg3.shape[1]) * 100
    print(f"  Sujeito {subj_test[i]}: {n_pos_subj:4d} positivos ({pct_pos_subj:5.2f}%)")

print(f"\n{'='*60}")
print(f"COMPARAÇÃO COM TREINO:")
print(f"{'='*60}")
print("Taxa de apneia no treino: ~6.9%")
print(f"Taxa de apneia no test (fillgap3): {pct_positives:.2f}%")

if pct_positives < 1.0:
    print("\n⚠️ ATENÇÃO: Taxa muito baixa! Considere ajustar threshold.")
elif pct_positives > 20.0:
    print("\n⚠️ ATENÇÃO: Taxa muito alta! Considere ajustar threshold.")
else:
    print("\n✓ Taxa parece razoável.")

ESTATÍSTICAS DAS PREDIÇÕES (FILL GAP 3s):

Total de predições: 396,000
  Positivas (1): 43,519 (10.99%)
  Negativas (0): 352,481 (89.01%)

Por sujeito:
  Sujeito 0: 1167 positivos ( 6.48%)
  Sujeito 1: 2232 positivos (12.40%)
  Sujeito 2:  938 positivos ( 5.21%)
  Sujeito 3:  200 positivos ( 1.11%)
  Sujeito 4: 1886 positivos (10.48%)
  Sujeito 5: 7488 positivos (41.60%)
  Sujeito 6: 1621 positivos ( 9.01%)
  Sujeito 7:  152 positivos ( 0.84%)
  Sujeito 8: 10025 positivos (55.69%)
  Sujeito 9:  787 positivos ( 4.37%)
  Sujeito 10:  301 positivos ( 1.67%)
  Sujeito 11: 1328 positivos ( 7.38%)
  Sujeito 12: 2854 positivos (15.86%)
  Sujeito 13:  387 positivos ( 2.15%)
  Sujeito 14:  861 positivos ( 4.78%)
  Sujeito 15:  403 positivos ( 2.24%)
  Sujeito 16: 1186 positivos ( 6.59%)
  Sujeito 17:  321 positivos ( 1.78%)
  Sujeito 18:  301 positivos ( 1.67%)
  Sujeito 19: 2381 positivos (13.23%)
  Sujeito 20: 6620 positivos (36.78%)
  Sujeito 21:   80 positivos ( 0.44%)

COMPARAÇÃO COM TREIN

In [38]:
# Salvar submission com threshold simples
sub_simple = save_submission_csv(
    subj_test,
    y_test_pred_simple,
    out_csv="submission_simple_threshold.csv"
)

Saved submission: submission_simple_threshold.csv
  Shape: (4400, 91) (esperado: 4400 linhas × 91 colunas)
  Total de janelas: 4400 (22 sujeitos × 200 janelas)


In [41]:
# Resumo: modelo e parâmetros usados na submissão
summary = {
    "model": "LGBMClassifier ensemble (5 folds GroupKFold)",
    "n_models": len(models),
    "n_features": len(feature_names),
    "threshold_simple": simple_threshold,       # 0.11
    "postprocess": "fill_short_gaps only",
    "fill_short_gaps_max_gap": 3,
    "smoothing": "none",
    "hysteresis": "none",
    "min_len": "not applied",
    "train_apnea_rate_pct": "~6.9%",
    "test_pred_pct_after_fillgap3": float((y_test_pred_simple_fg3.sum() / y_test_pred_simple_fg3.size) * 100),
    "submission_files": [
        "submission_simple_threshold.csv",
        "submission_simple_threshold_fillgap3.csv"
    ],
}
pd.Series(summary)



model                                LGBMClassifier ensemble (5 folds GroupKFold)
n_models                                                                        5
n_features                                                                     18
threshold_simple                                                             0.11
postprocess                                                  fill_short_gaps only
fill_short_gaps_max_gap                                                         3
smoothing                                                                    none
hysteresis                                                                   none
min_len                                                               not applied
train_apnea_rate_pct                                                        ~6.9%
test_pred_pct_after_fillgap3                                            10.989646
submission_files                [submission_simple_threshold.csv, submission_s...
dtype: object